# 🏦 JPMC Consumer Banking - Hands-on Lab
## Notebook 07: Storage Lifecycle Policies & Cortex Code

---

### What You'll Learn
- **Storage Lifecycle Policies** - Automatically archive cold data to reduce costs
- **Cortex Code (CoCo)** - AI-powered developer productivity tool walkthrough

### Why These Topics Matter for Banking
- Regulations require long data retention (7+ years) → lifecycle policies manage cost
- Developer productivity at scale → Cortex Code accelerates SQL/Python authoring

In [ ]:
-- Setup context
USE ROLE ACCOUNTADMIN;
USE DATABASE JPMC_CONSUMER_BANKING_HOL;
USE SCHEMA HOL_LAB;
USE WAREHOUSE HOL_XS_WH;

---
## Part 1: Storage Lifecycle Policies (Data Archival)

Storage Lifecycle Policies automatically move data between storage tiers based on age:
- **ACTIVE** tier - Full performance, highest cost
- **COOL** tier - Reduced cost (~40% savings), slightly higher access latency
- **COLD** tier - Lowest cost, highest access latency (for rarely accessed data)

### Use Case: Banking Regulatory Retention
Banks must retain transaction data for 7+ years. Active analytics only needs recent data. Archive older data to save costs while maintaining compliance.

In [ ]:
-- =============================================================================
-- STORAGE LIFECYCLE POLICY: Archive transaction data after 90 days to COOL
-- =============================================================================

CREATE OR REPLACE STORAGE LIFECYCLE POLICY HOL_TXN_ARCHIVE_POLICY
    AS (txn_date VARCHAR)
    RETURNS BOOLEAN ->
        TO_TIMESTAMP(txn_date) < DATEADD(DAY, -90, CURRENT_TIMESTAMP())
    ARCHIVE_TIER = COOL
    ARCHIVE_FOR_DAYS = 270
    COMMENT = 'Archive transactions older than 90 days to COOL tier';

In [ ]:
-- Attach the lifecycle policy to the TRANSACTIONS table
ALTER TABLE TRANSACTIONS 
    ADD STORAGE LIFECYCLE POLICY HOL_TXN_ARCHIVE_POLICY ON (TXN_DATE);

-- Verify policy attachment
SHOW STORAGE LIFECYCLE POLICIES;

-- Describe the policy
DESCRIBE STORAGE LIFECYCLE POLICY HOL_TXN_ARCHIVE_POLICY;

-- Note: Archived data is still queryable! Same SQL, same semantics.
-- The only difference is slightly higher latency for COOL/COLD tier access.

-- =========================================================================
-- RESTORE ARCHIVED DATA: Use CREATE TABLE ... FROM ARCHIVE OF
-- =========================================================================
-- To restore archived rows into a new table:
--
--   CREATE TABLE TRANSACTIONS_RESTORED
--     FROM ARCHIVE OF TRANSACTIONS
--     WHERE TO_TIMESTAMP(TXN_DATE) BETWEEN '2024-01-01' AND '2024-03-31';
--
-- This creates a new table with the archived rows restored to active storage.
-- =========================================================================


### Cortex Code Demo Prompts

Try these prompts in Cortex Code (accessible from Snowsight or the CoCo Desktop IDE):

**1. SQL Generation:**
```
Write a query to find the top 10 customers by total balance across all their accounts,
including their risk rating and number of delinquent loans
```

**2. Transformation:**
```
Create a Dynamic Table that calculates monthly customer churn rate 
by comparing active customers this month vs last month, grouped by segment
```

**3. Debugging:**
```
My COPY INTO is failing with "Number of columns in file does not match table".
The Parquet file has 12 columns but my table has 14 columns. How do I fix this?
```

**4. Pipeline Design:**
```
Help me design a Task DAG that:
1. Runs data quality checks every hour
2. If checks pass, refreshes the analytics tables
3. If checks fail, logs the error and skips the refresh
```

**5. Cost Optimization:**
```
Analyze my warehouse usage and suggest which queries should use 
which warehouse size for optimal cost/performance balance
```

## Summary

| Topic | Key Takeaway |
|-------|-------------|
| **Storage Lifecycle** | Automatically archive cold data → 40%+ cost savings with transparent access |
| **Iceberg Tables** | Open format enables multi-engine access without data duplication |
| **Catalog Integration** | Horizon REST Catalog lets Spark/Trino/Flink discover & read Iceberg tables |
| **Cortex Code** | AI-powered coding assistant accelerates SQL/Python development 3-5x |

---

**Next:** Proceed to `08_Spark_SPCS_Iceberg_Horizon.ipynb` to run Apache Spark in a container and query Iceberg tables without Snowflake compute.